This is a companion notebook for the book [Deep Learning with Python, Third Edition](https://www.manning.com/books/deep-learning-with-python-third-edition). For readability, it only contains runnable code blocks and section titles, and omits everything else in the book: text paragraphs, figures, and pseudocode.

**If you want to be able to follow what's going on, I recommend reading the notebook side by side with your copy of the book.**

The book's contents are available online at [deeplearningwithpython.io](https://deeplearningwithpython.io).

In [1]:
#!pip install keras keras-hub --upgrade -q

In [2]:
import os
os.environ["KERAS_BACKEND"] = "jax"

In [3]:
# @title
import os
from IPython.core.magic import register_cell_magic

@register_cell_magic
def backend(line, cell):
    current, required = os.environ.get("KERAS_BACKEND", ""), line.split()[-1]
    if current == required:
        get_ipython().run_cell(cell)
    else:
        print(
            f"This cell requires the {required} backend. To run it, change KERAS_BACKEND to "
            f"\"{required}\" at the top of the notebook, restart the runtime, and rerun the notebook."
        )

## **Language models** and the Transformer

이 장에서는 다음 내용을 다룹니다.

* 딥러닝 모델을 이용한 텍스트 생성 방법
* 영어를 스페인어로 번역하는 모델 학습
* 텍스트 모델링 문제를 위한 강력한 아키텍처, 트랜스포머

이전 장에서 텍스트 전처리 및 모델링의 기초를 다룬 후, 이 장에서는 기계 번역과 같은 좀 더 복잡한 언어 문제를 다룹니다. ChatGPT와 같은 제품에 적용되어 자연어 처리(NLP) 분야에 대한 투자를 촉발시킨 트랜스포머 모델에 대한 탄탄한 이해를 쌓아갈 것입니다.

### The language model

이전 장에서는 텍스트 데이터를 숫자 입력으로 변환하는 방법을 배우고, 이 숫자 표현을 사용하여 영화 리뷰를 분류했습니다. 하지만 텍스트 분류는 여러 면에서 매우 간단한 문제입니다. 이진 분류의 경우 하나의 부동 소수점 숫자만 출력하면 되고, N개 변수 분류의 경우에도 최악의 경우 N개의 숫자만 출력하면 됩니다.

그렇다면 질문 답변이나 번역과 같은 다른 텍스트 기반 작업은 어떨까요? 많은 실제 문제에서 우리는 주어진 입력에 대한 텍스트 출력을 생성할 수 있는 모델에 관심이 있습니다. 모델에 텍스트를 입력하기 위해 토크나이저와 임베딩이 필요했던 것처럼, 모델에서 텍스트를 출력하기 전에도 몇 가지 기술을 구축해야 합니다.

여기서 처음부터 시작할 필요는 없습니다. 텍스트를 자연스럽게 숫자로 표현하는 정수 시퀀스라는 개념을 계속 사용할 수 있습니다. 이전 장에서는 입력을 토큰으로 분할하고 각 토큰을 정수로 매핑하는 문자열 토큰화 방법을 다뤘습니다. 시퀀스를 역으로 토큰화하려면 정수를 다시 문자열 토큰으로 매핑하고 이들을 결합하면 됩니다. 이러한 접근 방식을 사용하면, 우리의 문제는 토큰의 정수 시퀀스를 예측할 수 있는 모델을 구축하는 것으로 귀결됩니다.

가장 간단한 방법은 가능한 모든 출력 정수 시퀀스 공간에 대해 직접 분류기를 학습시키는 것이지만, 간단한 계산만으로도 이것이 현실적으로 불가능하다는 것을 알 수 있습니다. 20,000개의 단어로 이루어진 어휘라면, 20,000^4, 즉 160경 개의 가능한 4단어 시퀀스가 존재하며, 우주의 원자 수보다 20단어로 이루어진 시퀀스의 수가 더 많습니다. 모든 출력 시퀀스를 고유한 분류기 출력으로 표현하려고 시도하는 것은 모델을 어떻게 설계하든 컴퓨팅 자원을 한계까지 밀어붙일 것입니다.

이러한 예측 문제를 실현 가능하게 만드는 실용적인 접근 방식은 한 번에 하나의 토큰 출력만 예측하는 모델을 구축하는 것입니다. 언어 모델은 가장 간단한 형태로, 단순하지만 심오한 확률 분포인 p(토큰|이전 토큰)을 학습하는 모델입니다. 특정 시점까지 관찰된 모든 토큰 시퀀스가 주어졌을 때, 언어 모델은 다음에 올 수 있는 모든 가능한 토큰에 대한 확률 분포를 출력하려고 시도합니다. 20,000개의 단어 어휘를 가진 모델은 20,000개의 출력만 예측하면 되지만, 다음 토큰을 반복적으로 예측함으로써 긴 텍스트 시퀀스를 생성할 수 있는 모델을 구축할 수 있습니다.

이를 좀 더 구체적으로 이해하기 위해, 문자열 시퀀스에서 다음 문자를 예측하는 간단한 언어 모델을 만들어 보겠습니다. 셰익스피어풍의 텍스트를 출력할 수 있는 작은 모델을 학습시켜 보겠습니다.

#### Training a Shakespeare language model

우선, 셰익스피어의 희곡과 소네트 모음집을 다운로드할 수 있습니다.

In [4]:
import keras

filename = keras.utils.get_file(
    origin=(
        "https://storage.googleapis.com/download.tensorflow.org/"
        "data/shakespeare.txt"
    ),
)
shakespeare = open(filename, "r").read()

1115394/1115394 ━━━━━━━━━━━━━━━━━━━━ 1s 1us/step


몇 가지 데이터를 살펴보겠습니다.

In [6]:
print(shakespeare[:250])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.



이 입력값을 기반으로 언어 모델을 구축하려면 원본 텍스트를 가공해야 합니다. 먼저, 시계열 분석에서 날씨 측정값을 처리했던 것처럼 데이터를 동일한 길이의 덩어리로 나누어 배치 처리하고 모델 학습에 사용합니다. 문자 단위 토크나이저를 사용하기 때문에 문자열 입력에 직접 덩어리 분할을 적용할 수 있습니다. 100자 문자열은 100개의 정수로 이루어진 시퀀스로 변환됩니다.

또한 각 입력값을 두 개의 개별 특징 시퀀스와 레이블 시퀀스로 분리합니다. 각 레이블 시퀀스는 입력 시퀀스에서 한 문자만큼 오프셋된 값입니다.

In [7]:
import tensorflow as tf

sequence_length = 100

def split_input(input, sequence_length):
    for i in range(0, len(input), sequence_length):
        yield input[i : i + sequence_length]

features = list(split_input(shakespeare[:-1], sequence_length))
labels = list(split_input(shakespeare[1:], sequence_length))
dataset = tf.data.Dataset.from_tensor_slices((features, labels))

(x, y) 입력 샘플을 살펴보겠습니다. 시퀀스의 각 위치에 대한 레이블은 시퀀스에서 다음 문자입니다.

In [8]:
x, y = next(dataset.as_numpy_iterator())
x[:50], y[:50]

(b'First Citizen:\nBefore we proceed any further, hear',
 b'irst Citizen:\nBefore we proceed any further, hear ')

이 입력을 정수 시퀀스로 매핑하기 위해 지난 장에서 살펴본 텍스트 벡터화 레이어를 다시 사용할 수 있습니다. 단어 수준 어휘 대신 문자 수준 어휘를 학습하려면 split 인수를 변경할 수 있습니다. 기본값인 "공백" 분할 대신 "문자"를 기준으로 분할합니다. 여기서는 표준화를 수행하지 않고 간단하게 유지하기 위해 대소문자를 유지하고 구두점은 변경하지 않고 그대로 전달합니다.

In [9]:
from keras import layers

tokenizer = layers.TextVectorization(
    standardize=None,
    split="character",
    output_sequence_length=sequence_length,
)
tokenizer.adapt(dataset.map(lambda text, labels: text))

어휘를 살펴보겠습니다.

In [10]:
vocabulary_size = tokenizer.vocabulary_size()
vocabulary_size

67

전체 원문을 처리하는 데 필요한 문자는 단 67개뿐입니다.

다음으로, 입력 텍스트에 토큰화 레이어를 적용할 수 있습니다. 마지막으로, 데이터셋을 섞고, 배치 처리하고, 캐싱하여 매 에포크마다 다시 계산할 필요가 없도록 할 수 있습니다.

In [11]:
dataset = dataset.map(
    lambda features, labels: (tokenizer(features), tokenizer(labels)),
    num_parallel_calls=8,
)
training_data = dataset.shuffle(10_000).batch(64).cache()

이제 모델링을 시작할 준비가 되었습니다.

간단한 언어 모델을 구축하기 위해, 우리는 과거의 모든 문자를 기반으로 특정 문자의 확률을 예측하고자 합니다. 이 책에서 살펴본 다양한 모델링 방법 중에서 RNN(순환 신경망)이 가장 적합합니다. 각 셀의 순환 상태를 통해 과거 문자에 대한 정보를 현재 문자의 레이블을 예측하는 데 활용할 수 있기 때문입니다. 또한 이전 장에서 살펴본 것처럼 임베딩을 사용하여 각 입력 문자를 고유한 256차원 벡터로 임베딩할 수도 있습니다.

모델의 크기를 줄이고 학습을 용이하게 하기 위해 순환 레이어는 하나만 사용하겠습니다. 어떤 순환 레이어든 사용할 수 있지만, 간단하게 GRU(지능형 러그 메모리)를 사용하겠습니다. GRU는 속도가 빠르고 LSTM(선형 러그 메모리)보다 내부 상태가 간단합니다.

In [12]:
embedding_dim = 256
hidden_dim = 1024

inputs = layers.Input(shape=(sequence_length,), dtype="int", name="token_ids")
x = layers.Embedding(vocabulary_size, embedding_dim)(inputs)
x = layers.GRU(hidden_dim, return_sequences=True)(x)
x = layers.Dropout(0.1)(x)
outputs = layers.Dense(vocabulary_size, activation="softmax")(x)
model = keras.Model(inputs, outputs)

모델 요약을 살펴보겠습니다.

In [13]:
model.summary(line_length=80)

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                      ┃ Output Shape             ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ token_ids (InputLayer)            │ (None, 100)              │             0 │
├───────────────────────────────────┼──────────────────────────┼───────────────┤
│ embedding (Embedding)             │ (None, 100, 256)         │        17,152 │
├───────────────────────────────────┼──────────────────────────┼───────────────┤
│ gru (GRU)                         │ (None, 100, 1024)        │     3,938,304 │
├───────────────────────────────────┼──────────────────────────┼───────────────┤
│ dropout (Dropout)                 │ (None, 100, 1024)        │             0 │
├───────────────────────────────────┼──────────────────────────┼───────────────┤
│ dense (Dense)                     │ (None, 100, 67)          │        68,675 │
└───────────────────────────────────┴──────────────────────────┴───────────────┘

 Total params: 4,024,131 (15.35 MB)

 Trainable params: 4,024,131 (15.35 MB)

 Non-trainable params: 0 (0.00 B)

이 모델은 어휘에 있는 모든 문자에 대해 소프트맥스 확률을 출력하며, 크로스엔트로피 손실 함수를 사용하여 컴파일합니다. 이 모델은 여전히 ​​분류 문제를 학습하는 것이지만, 시퀀스의 각 토큰에 대해 하나의 분류 예측을 수행합니다. 100개의 문자로 구성된 64개의 샘플 배치에 대해 6,400개의 개별 레이블을 예측합니다. 학습 중에 Keras에서 보고되는 손실 및 정확도 지표는 먼저 각 시퀀스별로, 그리고 두 번째로 각 배치별로 평균을 냅니다.

이제 언어 모델 학습을 시작해 보겠습니다.

In [14]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["sparse_categorical_accuracy"],
)
model.fit(training_data, epochs=20)

Epoch 1/20
175/175 ━━━━━━━━━━━━━━━━━━━━ 229s 1s/step - loss: 2.6839 - sparse_categorical_accuracy: 0.2800
Epoch 2/20
175/175 ━━━━━━━━━━━━━━━━━━━━ 303s 2s/step - loss: 1.9798 - sparse_categorical_accuracy: 0.4203
Epoch 3/20
175/175 ━━━━━━━━━━━━━━━━━━━━ 296s 2s/step - loss: 1.7101 - sparse_categorical_accuracy: 0.4925
Epoch 4/20
175/175 ━━━━━━━━━━━━━━━━━━━━ 296s 2s/step - loss: 1.5619 - sparse_categorical_accuracy: 0.5313
Epoch 5/20
175/175 ━━━━━━━━━━━━━━━━━━━━ 302s 2s/step - loss: 1.4723 - sparse_categorical_accuracy: 0.5540
Epoch 6/20
175/175 ━━━━━━━━━━━━━━━━━━━━ 293s 2s/step - loss: 1.4099 - sparse_categorical_accuracy: 0.5696
Epoch 7/20
175/175 ━━━━━━━━━━━━━━━━━━━━ 298s 2s/step - loss: 1.3596 - sparse_categorical_accuracy: 0.5826
Epoch 8/20
175/175 ━━━━━━━━━━━━━━━━━━━━ 293s 2s/step - loss: 1.3164 - sparse_categorical_accuracy: 0.5936
Epoch 9/20
175/175 ━━━━━━━━━━━━━━━━━━━━ 293s 2s/step - loss: 1.2771 - sparse_categorical_accuracy: 0.6039
Epoch 10/20
175/175 ━━━━━━━━━━━━━━━━━━━━ 305s 

20번의 에포크를 거친 후, 우리 모델은 입력 시퀀스에서 다음 문자를 약 70%의 확률로 예측할 수 있게 되었습니다.

#### Generating Shakespeare

이제 개별 토큰을 어느 정도 정확하게 예측할 수 있는 모델을 학습시켰으므로, 이를 사용하여 전체 예측 시퀀스를 외삽해 보고자 합니다. 이를 위해 모델을 반복문에서 호출하여, 한 시점의 모델 예측 출력을 다음 시점의 모델 입력으로 사용할 수 있습니다. 이러한 피드백 루프를 위해 구축된 모델을 자기회귀 모델이라고 합니다.

이러한 루프를 실행하려면 방금 학습시킨 모델을 약간 수정해야 합니다. 학습 과정에서는 모델이 100개의 토큰으로 구성된 고정된 시퀀스 길이만 처리했고, GRU 셀의 상태는 레이어를 호출할 때 암묵적으로 전달되었습니다. 하지만 생성 과정에서는 한 번에 하나의 출력 토큰만 예측하고 GRU 셀의 상태를 명시적으로 출력해야 합니다. 모델이 과거 입력 문자에 대해 인코딩한 모든 정보를 담고 있는 이 상태를 다음 호출 시점에 전달해야 합니다.

이제 한 번에 하나의 입력 문자만 처리하고 RNN 상태를 명시적으로 전달할 수 있는 모델을 만들어 보겠습니다. 이 모델은 입력과 출력이 약간 수정된 것을 제외하고는 동일한 계산 구조를 가지므로, 한 모델의 가중치를 다른 모델에 할당할 수 있습니다.

In [15]:
inputs = keras.Input(shape=(1,), dtype="int", name="token_ids")
input_state = keras.Input(shape=(hidden_dim,), name="state")

x = layers.Embedding(vocabulary_size, embedding_dim)(inputs)
x, output_state = layers.GRU(hidden_dim, return_state=True)(
    x, initial_state=input_state
)
outputs = layers.Dense(vocabulary_size, activation="softmax")(x)
generation_model = keras.Model(
    inputs=(inputs, input_state),
    outputs=(outputs, output_state),
)
generation_model.set_weights(model.get_weights())

이렇게 하면 루프를 통해 모델을 호출하여 출력 시퀀스를 예측할 수 있습니다. 그 전에, 문자에서 정수로 전환하고 프롬프트(새로운 토큰 예측을 시작하기 전에 모델에 입력할 텍스트 조각)를 선택하기 위해 명시적인 조회 테이블을 만들겠습니다.

In [16]:
tokens = tokenizer.get_vocabulary()
token_ids = range(vocabulary_size)
char_to_id = dict(zip(tokens, token_ids))
id_to_char = dict(zip(token_ids, tokens))

prompt = """
KING RICHARD III:
"""

응답 생성을 시작하려면 먼저 프롬프트를 사용하여 GRU의 내부 상태를 "준비"해야 합니다. 이를 위해 프롬프트를 토큰 단위로 모델에 입력합니다. 이렇게 하면 학습 중에 해당 프롬프트를 만났을 때 모델이 보게 될 정확한 RNN 상태를 계산할 수 있습니다.

프롬프트의 마지막 문자를 모델에 입력하면 상태 출력에 전체 프롬프트 시퀀스에 대한 정보가 포함됩니다. 최종 출력 예측값을 저장해 두면 나중에 생성된 응답의 첫 번째 문자를 선택하는 데 사용할 수 있습니다.

In [17]:
input_ids = [char_to_id[c] for c in prompt]
state = keras.ops.zeros(shape=(1, hidden_dim))
for token_id in input_ids:
    inputs = keras.ops.expand_dims([token_id], axis=0)
    predictions, state = generation_model.predict((inputs, state), verbose=0)

이제 모델이 새로운 출력 시퀀스를 예측하도록 할 준비가 되었습니다. 원하는 길이까지 반복문을 사용하여 모델이 예측한 다음 문자 중 가장 가능성이 높은 문자를 지속적으로 선택하고, 이를 모델에 입력한 다음, 새로운 RNN 상태를 저장합니다. 이러한 방식으로 전체 시퀀스를 한 번에 하나의 토큰씩 예측할 수 있습니다.

In [18]:
import numpy as np

generated_ids = []
max_length = 250
for i in range(max_length):
    next_char = int(np.argmax(predictions, axis=-1)[0])
    generated_ids.append(next_char)
    inputs = keras.ops.expand_dims([next_char], axis=0)
    predictions, state = generation_model.predict((inputs, state), verbose=0)

모델이 예측한 결과를 확인하기 위해 출력된 정수 시퀀스를 문자열로 변환해 보겠습니다. 입력값을 토큰화 해제하려면 모든 토큰 ID를 문자열로 매핑하고 이를 결합하면 됩니다.

다음과 같은 출력이 나타납니다.

In [19]:
output = "".join([id_to_char[token_id] for token_id in generated_ids])
print(prompt + output)


KING RICHARD III:
Say, Say that thou hast horrow to me against the gods
That would be gone and honesty of this famous life?

KING RICHARD III:
Say, Say that thou hast horrow to me against the gods
That would be gone and honesty of this famous life?

KING RICHARD III:



아직 다음 대비극을 만들어내지는 못했지만, 최소한의 데이터셋으로 2분 정도 학습시킨 결과치고는 나쁘지 않습니다. 이 간단한 예제의 목적은 언어 모델 설정의 강력함을 보여주는 것입니다. 우리는 한 번에 한 글자씩 추측하는 좁은 문제로 모델을 학습시켰지만, 이 모델을 훨씬 더 광범위한 문제, 즉 셰익스피어 작품처럼 끝없이 이어지는 텍스트 응답을 생성하는 데 활용했습니다.

이러한 학습 설정이 가능한 이유는 순환 신경망이 시퀀스에서 정보를 앞으로만 전달하기 때문이라는 점에 유의해야 합니다. 원한다면 GRU 레이어를 양방향(GRU(...))으로 바꿔보세요. 학습 정확도가 즉시 99%를 넘어설 것이고, 생성 기능은 완전히 작동을 멈출 것입니다. 학습 과정에서 우리 모델은 매 학습 단계마다 전체 시퀀스를 접합니다. 만약 시퀀스에서 다음 토큰의 정보가 현재 토큰의 예측에 영향을 미치도록 "꼼수"를 부린다면, 문제는 아주 쉬워지는 것입니다.

이러한 언어 모델링 설정은 텍스트 영역의 수많은 문제에 대한 기본 토대가 됩니다. 또한 이 책에서 지금까지 살펴본 다른 모델링 문제들과 비교했을 때 다소 독특한 측면도 있습니다. 단순히 `model.predict()`를 호출하는 것만으로는 원하는 출력을 얻을 수 없습니다. 추론 시점에만 존재하는 복잡한 루프와 상당한 양의 로직이 있기 때문입니다! RNN 셀의 상태 순환은 학습과 추론 모두에서 발생하지만, 학습 과정에서는 모델이 예측한 레이블을 다시 입력으로 사용하는 경우는 없습니다.

### Sequence-to-sequence learning

언어 모델 개념을 확장하여 중요한 문제인 기계 번역을 다뤄 보겠습니다. 번역은 시퀀스-투-시퀀스 모델링(seq2seq)이라고 불리는 모델링 문제의 한 유형입니다. 원문을 고정된 입력 시퀀스로 받아 번역된 텍스트 시퀀스를 결과로 생성하는 모델을 구축하는 것이 목표입니다. 질의응답 또한 대표적인 시퀀스-투-시퀀스 문제입니다.

시퀀스-투-시퀀스 모델의 기본적인 구조는 그림 15.1에 나와 있습니다. 학습 과정에서는 다음과 같은 단계가 진행됩니다.

* 인코더 모델은 원문 시퀀스를 중간 표현으로 변환합니다.
* 디코더는 앞서 살펴본 언어 모델링 방식을 사용하여 학습됩니다. 디코더는 이전의 모든 대상 토큰과 인코더가 생성한 원문 시퀀스의 표현을 이용하여 대상 시퀀스의 다음 토큰을 재귀적으로 예측합니다.

추론 단계에서는 대상 시퀀스에 접근할 수 없습니다. 처음부터 대상 시퀀스를 예측해야 합니다. 셰익스피어 생성기에서 했던 것처럼 토큰을 하나씩 순차적으로 생성해 보겠습니다.

* 인코더에서 인코딩된 소스 시퀀스를 얻습니다.
* 디코더는 인코딩된 소스 시퀀스와 초기 "시드" 토큰(예: "start" 문자열)을 사용하여 시퀀스의 첫 번째 실제 토큰을 예측합니다.
* 지금까지 예측된 시퀀스는 디코더에 반복적으로 입력되어 "end" 토큰(예: "end" 문자열)이 생성될 때까지 계속됩니다.

<p style="text-align:center">
<img src="https://deeplearningwithpython.io/images/ch15/seq2seq-learning.0e1e1c31.png" width="600"><br>Figure 15.1: Sequence-to-sequence learning: the source sequence is processed by the encoder and is then sent to the decoder. The decoder looks at the target sequence so far and predicts the target sequence offset by one step in the future. During inference, we generate one target token at a time and feed it back into the decoder.</p>

서열 대 서열 번역 모델을 구축해 봅시다.

#### English-to-Spanish translation

우리는 영어-스페인어 번역 데이터셋을 사용할 것입니다. 다운로드해 봅시다.

In [20]:
import pathlib

zip_path = keras.utils.get_file(
    origin=(
        "http://storage.googleapis.com/download.tensorflow.org/data/spa-eng.zip"
    ),
    fname="spa-eng",
    extract=True,
)
text_path = pathlib.Path(zip_path) / "spa-eng" / "spa.txt"

2638744/2638744 ━━━━━━━━━━━━━━━━━━━━ 3s 1us/step


In [23]:
text_path

WindowsPath('C:/Users/admin/.keras/datasets/spa-eng/spa-eng/spa.txt')

텍스트 파일에는 각 줄마다 하나의 예시가 있습니다. 영어 문장 다음에 탭 문자가 오고, 그 다음에 해당 스페인어 문장이 옵니다. 이 파일을 분석해 보겠습니다.

In [25]:
with open(text_path, encoding="utf-8") as f:
    lines = f.read().split("\n")[:-1]
text_pairs = []
for line in lines:
    english, spanish = line.split("\t")
    spanish = "[start] " + spanish + " [end]"
    text_pairs.append((english, spanish))

저희의 텍스트 쌍은 다음과 같습니다.

In [26]:
import random
random.choice(text_pairs)

('Tom is an unknown artist.', '[start] Tom es un artista desconocido. [end]')

이제 이 데이터셋들을 섞어서 일반적인 학습, 검증, 테스트 세트로 나누어 보겠습니다.

In [27]:
import random

random.shuffle(text_pairs)
val_samples = int(0.15 * len(text_pairs))
train_samples = len(text_pairs) - 2 * val_samples
train_pairs = text_pairs[:train_samples]
val_pairs = text_pairs[train_samples : train_samples + val_samples]
test_pairs = text_pairs[train_samples + val_samples :]

다음으로, 영어와 스페인어 각각에 대한 두 개의 별도 텍스트 벡터화 레이어를 준비해 보겠습니다. 문자열 전처리 방식을 사용자 정의해야 합니다.

* 삽입한 "[start]"와 "[end]" 토큰을 유지해야 합니다. 기본적으로 [ ] 문자는 제거되지만, "start"라는 단어와 시작 토큰 "[start]"를 구분하기 위해 이 문자들을 남겨두어야 합니다.
* 언어마다 구두점 표기법이 다릅니다! 스페인어 텍스트 벡터화 레이어에서 구두점을 제거하려면 ¿ 문자도 함께 제거해야 합니다.

참고로, 실제 번역 모델에서는 구두점을 제거하는 대신 별도의 토큰으로 처리하여 구두점이 있는 문장도 생성할 수 있도록 해야 합니다. 하지만 여기서는 간단하게 모든 구두점을 제거하겠습니다.

In [28]:
import string
import re

strip_chars = string.punctuation + "¿"
strip_chars = strip_chars.replace("[", "")
strip_chars = strip_chars.replace("]", "")

def custom_standardization(input_string):
    lowercase = tf.strings.lower(input_string)
    return tf.strings.regex_replace(
        lowercase, f"[{re.escape(strip_chars)}]", ""
    )

vocab_size = 15000
sequence_length = 20

english_tokenizer = layers.TextVectorization(
    max_tokens=vocab_size,
    output_mode="int",
    output_sequence_length=sequence_length,
)
spanish_tokenizer = layers.TextVectorization(
    max_tokens=vocab_size,
    output_mode="int",
    output_sequence_length=sequence_length + 1,
    standardize=custom_standardization,
)
train_english_texts = [pair[0] for pair in train_pairs]
train_spanish_texts = [pair[1] for pair in train_pairs]
english_tokenizer.adapt(train_english_texts)
spanish_tokenizer.adapt(train_spanish_texts)

마지막으로, 데이터를 `tf.data` 파이프라인으로 변환할 수 있습니다. 이 파이프라인은 `inputs`와 `spanish` 두 개의 키를 가진 딕셔너리 `(inputs, target, sample_weights)` 튜플을 반환하도록 설계되었습니다. `inputs`는 토큰화된 영어 문장 `english`와 `spanish`를 포함하는 딕셔너리이고, `target`은 한 단계 앞선 스페인어 문장의 오프셋 값입니다. `sample_weights`는 Keras에게 손실과 메트릭을 계산할 때 사용할 레이블을 지정하는 데 사용됩니다. 출력 번역문의 길이는 모두 같지 않으며, 일부 레이블 시퀀스는 0으로 채워집니다. 우리는 실제 번역된 텍스트를 나타내는 0이 아닌 레이블에 대한 예측만 중요하게 생각합니다.

이는 방금 구축한 생성 모델에서 설정한 "오프 바이 원(off by one)" 레이블과 동일하며, 고정된 인코더 입력이 추가된 것입니다. 인코더 입력은 모델에서 별도로 처리됩니다.

In [29]:
batch_size = 64

def format_dataset(eng, spa):
    eng = english_tokenizer(eng)
    spa = spanish_tokenizer(spa)
    features = {"english": eng, "spanish": spa[:, :-1]}
    labels = spa[:, 1:]
    sample_weights = labels != 0
    return features, labels, sample_weights

def make_dataset(pairs):
    eng_texts, spa_texts = zip(*pairs)
    eng_texts = list(eng_texts)
    spa_texts = list(spa_texts)
    dataset = tf.data.Dataset.from_tensor_slices((eng_texts, spa_texts))
    dataset = dataset.batch(batch_size)
    dataset = dataset.map(format_dataset, num_parallel_calls=4)
    return dataset.shuffle(2048).cache()

train_ds = make_dataset(train_pairs)
val_ds = make_dataset(val_pairs)

다음은 저희 데이터셋 출력 결과입니다.

In [30]:
inputs, targets, sample_weights = next(iter(train_ds))
print(inputs["english"].shape)

(64, 20)


In [31]:
print(inputs["spanish"].shape)

(64, 20)


In [32]:
print(targets.shape)

(64, 20)


In [33]:
print(sample_weights.shape)

(64, 20)


이제 데이터가 준비되었으니, 모델을 구축할 차례입니다.

#### Sequence-to-sequence learning with RNNs

앞서 언급한 트윈 인코더/디코더 설정을 시도하기 전에 더 간단한 옵션부터 살펴보겠습니다. RNN을 사용하여 하나의 시퀀스를 다른 시퀀스로 변환하는 가장 쉽고 단순한 방법은 각 시간 단계에서 RNN의 출력을 유지하고 이를 기반으로 출력 토큰을 예측하는 것입니다. Keras에서는 다음과 같이 표현할 수 있습니다.

```
inputs = keras.Input(shape=(sequence_length,), dtype="int32")
x = layers.Embedding(input_dim=vocab_size, output_dim=128)(inputs)
x = layers.LSTM(32, return_sequences=True)(x)
outputs = layers.Dense(vocab_size, activation="softmax")(x)
model = keras.Model(inputs, outputs)
```

하지만 이 접근 방식에는 치명적인 문제가 있습니다. RNN은 단계적으로 작동하기 때문에 소스 시퀀스의 0번째부터 N번째 토큰만 사용하여 타겟 시퀀스의 N번째 토큰을 예측합니다. "가방을 너에게 가져다 줄게"라는 문장을 스페인어로 번역한다고 가정해 보겠습니다. 스페인어로는 "Te traeré la bolsa"가 되는데, 번역의 첫 단어인 "Te"는 영어 원문의 "you"에 해당합니다. 원문의 마지막 단어를 보지 않고는 번역의 첫 단어를 출력할 방법이 없습니다!

인간 번역가라면 번역을 시작하기 전에 원문 전체를 읽을 것입니다. 특히 단어 순서가 매우 다른 언어를 다룰 때는 더욱 중요합니다. 그리고 이것이 바로 표준 시퀀스-투-시퀀스 모델이 하는 일입니다. 적절한 시퀀스-투-시퀀스 구성(그림 15.2 참조)에서는 먼저 인코더 RNN을 사용하여 전체 소스 시퀀스를 소스 텍스트의 단일 표현으로 변환합니다. 이는 RNN의 최종 출력일 수도 있고, 또는 최종 내부 상태 벡터일 수도 있습니다. 우리는 이 표현을 언어 모델 설정에서 디코더 RNN의 초기 상태로 사용할 수 있습니다. 셰익스피어 생성기에서 사용했던 것처럼 초기 상태를 0으로 설정하는 대신 말이죠. 이 디코더는 초기 RNN 상태에서 얻은 영어 시퀀스에 대한 모든 정보를 바탕으로 현재 번역어가 주어졌을 때 스페인어 번역어의 다음 단어를 예측하도록 학습합니다.

<p style="text-align:center">
<img src="https://deeplearningwithpython.io/images/ch15/seq2seq-rnn.ec377d3b.png" width="600"><br>Figure 15.2: A sequence-to-sequence RNN: an RNN encoder is used to produce a vector that encodes the entire source sequence, which is used as the initial state for an RNN decoder.</p>

GRU 기반 인코더와 디코더를 사용하여 Keras로 구현해 보겠습니다. 먼저 인코더부터 시작해 보죠. 인코더 시퀀스에서는 실제로 토큰을 예측하지 않으므로, 모델이 시퀀스 끝부분의 정보를 시작 부분으로 전달하는 방식으로 "꼼수"를 부릴 필요가 없습니다. 오히려 이렇게 하는 것이 좋습니다. 소스 시퀀스를 풍부하게 표현하고 싶기 때문입니다. 양방향 레이어를 사용하면 이를 구현할 수 있습니다.

In [34]:
embed_dim = 256
hidden_dim = 1024

source = keras.Input(shape=(None,), dtype="int32", name="english")
x = layers.Embedding(vocab_size, embed_dim, mask_zero=True)(source)
rnn_layer = layers.GRU(hidden_dim)
rnn_layer = layers.Bidirectional(rnn_layer, merge_mode="sum")
encoder_output = rnn_layer(x)

다음으로 디코더를 추가해 보겠습니다. 디코더는 인코딩된 원문 문장을 초기 상태로 받는 간단한 GRU 레이어입니다. 그 위에 각 출력 단계에서 스페인어 어휘에 대한 확률 분포를 생성하는 Dense 레이어를 추가합니다. 여기서는 이전 내용만을 기반으로 다음 토큰을 예측해야 하므로 양방향 RNN을 사용하면 손실 함수가 너무 단순해져 학습이 제대로 이루어지지 않을 수 있습니다.

In [35]:
target = keras.Input(shape=(None,), dtype="int32", name="spanish")
x = layers.Embedding(vocab_size, embed_dim, mask_zero=True)(target)
rnn_layer = layers.GRU(hidden_dim, return_sequences=True)
x = rnn_layer(x, initial_state=encoder_output)
x = layers.Dropout(0.5)(x)
target_predictions = layers.Dense(vocab_size, activation="softmax")(x)
seq2seq_rnn = keras.Model([source, target], target_predictions)

seq2seq 모델 전체를 살펴보겠습니다.

In [36]:
seq2seq_rnn.summary(line_length=80)

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)          ┃ Output Shape      ┃     Param # ┃ Connected to       ┃
┡━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━┩
│ english (InputLayer)  │ (None, None)      │           0 │ -                  │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ spanish (InputLayer)  │ (None, None)      │           0 │ -                  │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ embedding_2           │ (None, None, 256) │   3,840,000 │ english[0][0]      │
│ (Embedding)           │                   │             │                    │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ not_equal (NotEqual)  │ (None, None)      │           0 │ english[0][0]      │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ embedding_3           │ (None, None, 256) │   3,840,000 │ spanish[0][0]      │
│ (Embedding)           │                   │             │                    │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ bidirectional         │ (None, 1024)      │   7,876,608 │ embedding_2[0][0], │
│ (Bidirectional)       │                   │             │ not_equal[0][0]    │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ gru_3 (GRU)           │ (None, None,      │   3,938,304 │ embedding_3[0][0], │
│                       │ 1024)             │             │ bidirectional[0][… │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ dropout_1 (Dropout)   │ (None, None,      │           0 │ gru_3[0][0]        │
│                       │ 1024)             │             │                    │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ dense_2 (Dense)       │ (None, None,      │  15,375,000 │ dropout_1[0][0]    │
│                       │ 15000)            │             │                    │
└───────────────────────┴───────────────────┴─────────────┴────────────────────┘

 Total params: 34,869,912 (133.02 MB)

 Trainable params: 34,869,912 (133.02 MB)

 Non-trainable params: 0 (0.00 B)

모델과 데이터가 모두 준비되었습니다. 이제 번역 모델 학습을 시작할 수 있습니다.

In [37]:
seq2seq_rnn.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    weighted_metrics=["accuracy"],
)
seq2seq_rnn.fit(train_ds, epochs=15, validation_data=val_ds)

Epoch 1/15
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 1460s 1s/step - accuracy: 0.3559 - loss: 3.6825 - val_accuracy: 0.4993 - val_loss: 2.4945
Epoch 2/15
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 1679s 1s/step - accuracy: 0.5348 - loss: 2.2610 - val_accuracy: 0.5948 - val_loss: 1.8879
Epoch 3/15
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 2408s 2s/step - accuracy: 0.6220 - loss: 1.6332 - val_accuracy: 0.6306 - val_loss: 1.6870
Epoch 4/15
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 2444s 2s/step - accuracy: 0.6804 - loss: 1.2434 - val_accuracy: 0.6445 - val_loss: 1.6138
Epoch 5/15
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 2468s 2s/step - accuracy: 0.7277 - loss: 0.9885 - val_accuracy: 0.6524 - val_loss: 1.6239
Epoch 6/15
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 2403s 2s/step - accuracy: 0.7630 - loss: 0.8206 - val_accuracy: 0.6555 - val_loss: 1.6464
Epoch 7/15
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 2332s 2s/step - accuracy: 0.7877 - loss: 0.7152 - val_accuracy: 0.6576 - val_loss: 1.6736
Epoch 8/15
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 2330s 2s/step - accuracy: 0.8046 -

우리는 학습 과정에서 검증 세트 성능을 모니터링하는 간단한 방법으로 정확도를 선택했습니다. 65%의 정확도를 얻었는데, 이는 모델이 스페인어 문장에서 다음 단어를 평균 65%의 확률로 정확하게 예측한다는 의미입니다. 하지만 실제로는 다음 토큰 정확도가 기계 번역 모델에 적합한 지표는 아닙니다. 특히, 토큰 N+1을 예측할 때 0부터 N까지의 올바른 목표 토큰을 이미 알고 있다는 가정을 전제로 하기 때문입니다. 실제 추론 과정에서는 목표 문장을 처음부터 생성해야 하므로 이전에 생성된 토큰이 100% 정확하다고 가정할 수 없습니다. 실제 기계 번역 시스템을 개발할 때는 더욱 신중하게 지표를 설계해야 합니다. BLEU 점수와 같은 표준 지표는 기계 번역된 텍스트와 고품질 참조 번역 세트 간의 유사도를 측정하며, 약간의 순서 불일치를 허용할 수 있습니다.

마지막으로, 우리의 모델을 사용하여 추론을 수행해 보겠습니다. 테스트 세트에서 몇 개의 문장을 선택하여 모델이 어떻게 번역하는지 확인해 보겠습니다. 먼저 시드 토큰인 "start"를 인코딩된 영어 원문과 함께 디코더 모델에 입력합니다. 다음 토큰 예측값을 얻고, 이를 디코더에 반복적으로 다시 입력하면서 매 반복마다 새로운 목표 토큰을 하나씩 추출합니다. 이 과정은 "end"에 도달하거나 최대 문장 길이에 이를 때까지 계속됩니다.

In [38]:
import numpy as np

spa_vocab = spanish_tokenizer.get_vocabulary()
spa_index_lookup = dict(zip(range(len(spa_vocab)), spa_vocab))

def generate_translation(input_sentence):
    tokenized_input_sentence = english_tokenizer([input_sentence])
    decoded_sentence = "[start]"
    for i in range(sequence_length):
        tokenized_target_sentence = spanish_tokenizer([decoded_sentence])
        inputs = [tokenized_input_sentence, tokenized_target_sentence]
        next_token_predictions = seq2seq_rnn.predict(inputs, verbose=0)
        sampled_token_index = np.argmax(next_token_predictions[0, i, :])
        sampled_token = spa_index_lookup[sampled_token_index]
        decoded_sentence += " " + sampled_token
        if sampled_token == "[end]":
            break
    return decoded_sentence

test_eng_texts = [pair[0] for pair in test_pairs]
for _ in range(5):
    input_sentence = random.choice(test_eng_texts)
    print("-")
    print(input_sentence)
    print(generate_translation(input_sentence))

-
Let's get together and talk it over.
[start] [UNK] y [UNK] [end]
-
Both my sisters are teachers.
[start] mis dos hermanas son profesores [end]
-
They require extra help.
[start] ellos [UNK] ayuda de su ayuda [end]
-
There's little chance of keeping slim, unless you stick to a diet.
[start] hay muchas clases de que [UNK] a un hablante nativo para no [UNK] [end]
-
He is in the habit of eating only two meals a day.
[start] Él está a la misma día para comer dos veces al día [end]


최종 모델 가중치는 가중치의 무작위 초기화와 입력 데이터의 무작위 섞기에 따라 달라지므로 정확한 번역 결과는 실행할 때마다 다를 수 있습니다. 결과는 다음과 같습니다.

저희 모델은 기본적인 오류를 많이 범하지만, 장난감 모델치고는 꽤 잘 작동합니다.

이 추론 방식은 매우 간단하지만, 새로운 단어를 샘플링할 때마다 전체 소스 문장과 생성된 전체 목표 문장을 다시 처리하기 때문에 비효율적입니다. 실제 응용 프로그램에서는 변경되지 않은 상태를 다시 계산하지 않도록 주의해야 합니다. 디코더에서 새로운 토큰을 예측하는 데 필요한 것은 현재 토큰과 이전 RNN 상태뿐이며, 이는 각 반복 전에 캐시할 수 있습니다.

이 장난감 모델을 개선할 수 있는 방법은 많습니다. 인코더와 디코더 모두에 깊은 순환 레이어 스택을 사용하거나, LSTM과 같은 다른 RNN 레이어를 시도해 볼 수도 있습니다. 하지만 이러한 수정 외에도 시퀀스-투-시퀀스 학습에 대한 RNN 접근 방식에는 몇 가지 근본적인 한계가 있습니다.

* 소스 시퀀스 표현은 인코더 상태 벡터에 완전히 저장되어야 하므로 번역할 수 있는 문장의 크기와 복잡성이 크게 제한됩니다.
* RNN은 과거 정보를 점진적으로 잊어버리는 경향이 있어 매우 긴 시퀀스를 처리하는 데 어려움을 겪습니다. 시퀀스의 100번째 토큰에 도달할 때쯤이면 시퀀스 시작 부분에 대한 정보가 거의 남아 있지 않습니다.

순환 신경망(RNN)은 2010년대 중반 시퀀스-투-시퀀스 학습을 지배했습니다. 2017년경의 Google 번역은 방금 만든 것과 유사한 구조로 7개의 대형 LSTM 레이어를 쌓아서 구동되었습니다. 그러나 이러한 RNN의 한계로 인해 연구자들은 결국 트랜스포머(Transformer)라고 불리는 새로운 유형의 시퀀스 모델을 개발하게 되었습니다.